In [40]:
import pandas as pd
import pickle
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.model_selection import train_test_split

In [41]:
df_final = pd.read_csv('../../datas/dataset_final.csv')
print(f"Data Loaded: {df_final.shape}")

Data Loaded: (21853, 61)


In [42]:
print(df_final.columns.tolist())

['User_ID', 'Age_x', 'Gender_x', 'Height_cm_x', 'Initial_Weight_kg_x', 'Initial_BMI_x', 'BMI_Category', 'Body_Fat_Category_x', 'Body_Fat_Percentage', 'Goal_x', 'Workout_Frequency_x', 'Average_Duration_Minutes_x', 'level_x', 'Badminton_x', 'Football_x', 'Basketball_x', 'Tennis_x', 'Volleyball_x', 'Table_Tennis_x', 'Swim_x', 'Age_y', 'Gender_y', 'Height_cm_y', 'Initial_Weight_kg_y', 'Initial_BMI_y', 'BMI_Category_x', 'Body_Fat_Category_y', 'Body_Fat_Percentage_x', 'Goal_y', 'Workout_Frequency_y', 'Average_Duration_Minutes_y', 'level_y', 'Badminton_y', 'Football_y', 'Basketball_y', 'Tennis_y', 'Volleyball_y', 'Table_Tennis_y', 'Swim_y', 'Week', 'Weight_kg', 'BMI', 'Body_Fat_Percentage_y', 'Daily_Calories', 'Daily_Water_ml', 'Target_Protein_g', 'Target_Carbs_g', 'Target_Fat_g', 'Limit_Sugar_g', 'Target_Fiber_g', 'Limit_Cholesterol_mg', 'Target_Calcium_mg', 'Meal_Frequency', 'BMI_Category_y', 'Day', 'Muscle Group', 'Exercise Name', 'Equipment', 'Sets', 'Reps', 'Instructions']


In [43]:
cols_profile = ['User_ID', 'Age_x', 'Gender_x', 'Initial_Weight_kg_x', 'Goal_x', 'Workout_Frequency_x', 'level_x']
df_profiles = df_final[cols_profile].drop_duplicates().reset_index(drop=True)

# Preprocessing (Encoding & Scaling WAJIB untuk Evaluasi yang adil)
le_gender = LabelEncoder()
le_goal = LabelEncoder()
le_level = LabelEncoder()

df_profiles['Gender_Encoded'] = le_gender.fit_transform(df_profiles['Gender_x'])
df_profiles['Goal_Encoded'] = le_goal.fit_transform(df_profiles['Goal_x'])
df_profiles['level_Encoded'] = le_level.fit_transform(df_profiles['level_x'])

features_knn = ['Goal_Encoded', 'Workout_Frequency_x', 'level_Encoded', 'Gender_Encoded', 'Age_x', 'Initial_Weight_kg_x']

# Scaling 0-1 agar berat badan tidak mendominasi
scaler = MinMaxScaler()
features_scaled = scaler.fit_transform(df_profiles[features_knn])
df_features_scaled = pd.DataFrame(features_scaled, columns=features_knn)

# ==========================================
# 2. LOGIKA EVALUASI (Train-Test Split)
# ==========================================
# Skenario:
# - Train Set = Database User yang tersedia (Kolam Pencarian)
# - Test Set  = User Baru yang sedang mencari teman
X_train, X_test, idx_train, idx_test = train_test_split(
    df_features_scaled, df_profiles.index, test_size=0.2, random_state=42
)

# Latih Model pada Database (Train)
knn = NearestNeighbors(n_neighbors=1, metric='euclidean')
knn.fit(X_train)

,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",1
,"radius radius: float, default=1.0Range of parameter space to use by default for :meth:`radius_neighbors`queries.",1.0
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'auto'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'euclidean'
,"p p: float (positive), default=2Parameter for the Minkowski metric fromsklearn.metrics.pairwise.pairwise_distances. When p = 1, this isequivalent to using manhattan_distance (l1), and euclidean_distance(l2) for p = 2. For arbitrary p, minkowski_distance (l_p) is used.",2
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None


In [44]:
print("--- MENGHITUNG METRIK EVALUASI ---")
distances, indices = knn.kneighbors(X_test)
eval_data = []

for i in range(len(X_test)):
    idx_user_asli = idx_test[i]
    idx_teman_rekomendasi = idx_train[indices[i][0]]
    
    user = df_profiles.iloc[idx_user_asli]
    teman = df_profiles.iloc[idx_teman_rekomendasi]
    
    # Hitung Skor Kecocokan Baris ini
    # Kita beri bobot: Goal & Level & Freq itu KRUSIAL (Wajib sama)
    is_goal_same = (user['Goal_x'] == teman['Goal_x'])
    is_level_same = (user['level_x'] == teman['level_x'])
    is_freq_same = (user['Workout_Frequency_x'] == teman['Workout_Frequency_x'])
    is_gender_same = (user['Gender_x'] == teman['Gender_x'])
    
    selisih_umur = abs(user['Age_x'] - teman['Age_x'])
    selisih_berat = abs(user['Initial_Weight_kg_x'] - teman['Initial_Weight_kg_x'])
    
    eval_data.append({
        'User_ID': user['User_ID'],
        'User_Goal': user['Goal_x'],
        'Teman_Goal': teman['Goal_x'],
        'Sama_Goal': is_goal_same,
        'Sama_Level': is_level_same,
        'Sama_Freq': is_freq_same,
        'Sama_Gender': is_gender_same,
        'Selisih_Umur': selisih_umur,
        'Selisih_Berat': selisih_berat
    })

df_eval = pd.DataFrame(eval_data)

# ==========================================
# 4. LAPORAN AKHIR (REPORT CARD)
# ==========================================

print("\n" + "="*40)
print("     RAPOR KINERJA MODEL REKOMENDASI")
print("="*40)

# A. Metrik Utama (Hit Rate)
# Berapa persen user yang mendapatkan teman dengan kriteria UTAMA yang sama?
score_goal = df_eval['Sama_Goal'].mean() * 100
score_level = df_eval['Sama_Level'].mean() * 100
score_freq = df_eval['Sama_Freq'].mean() * 100
score_gender = df_eval['Sama_Gender'].mean() * 100

print(f"\n[A] KETEPATAN KATEGORI (Target: >80%)")
print(f"1. Goal Sama       : {score_goal:.2f}%  {'✅ Aman' if score_goal > 80 else '⚠️ Perlu Cek'}")
print(f"2. Level Sama      : {score_level:.2f}% {'✅ Aman' if score_level > 80 else '⚠️ Perlu Cek'}")
print(f"3. Frekuensi Sama  : {score_freq:.2f}%")
print(f"4. Gender Sama     : {score_gender:.2f}%")

# B. Metrik Numerik (Mean Absolute Error)
# Seberapa jauh bedanya umur dan berat badan?
print(f"\n[B] RATA-RATA PENYIMPANGAN")
print(f"1. Beda Umur       : Rata-rata selisih {df_eval['Selisih_Umur'].mean():.1f} tahun")
print(f"2. Beda Berat      : Rata-rata selisih {df_eval['Selisih_Berat'].mean():.1f} kg")

# C. Evaluasi Per Segmen (Penting!)
# Cek apakah model berat sebelah (bias) ke Goal tertentu
print(f"\n[C] PERFORMA PER TIPE GOAL")
segment_group = df_eval.groupby('User_Goal')['Sama_Goal'].agg(['count', 'mean'])
segment_group['Akurasi (%)'] = (segment_group['mean'] * 100).round(2)
print(segment_group[['count', 'Akurasi (%)']])

# D. Cek Kasus Terburuk (Debugging)
# Tampilkan 3 rekomendasi dengan selisih berat badan paling jauh
print(f"\n[D] 3 REKOMENDASI PALING MELESET (Berdasarkan Beda Berat)")
worst_cases = df_eval.sort_values('Selisih_Berat', ascending=False).head(3)
print(worst_cases[['User_Goal', 'Teman_Goal', 'Selisih_Berat', 'Sama_Level']].to_string(index=False))

print("\n" + "="*40)

--- MENGHITUNG METRIK EVALUASI ---

     RAPOR KINERJA MODEL REKOMENDASI

[A] KETEPATAN KATEGORI (Target: >80%)
1. Goal Sama       : 95.00%  ✅ Aman
2. Level Sama      : 95.00% ✅ Aman
3. Frekuensi Sama  : 55.00%
4. Gender Sama     : 100.00%

[B] RATA-RATA PENYIMPANGAN
1. Beda Umur       : Rata-rata selisih 3.9 tahun
2. Beda Berat      : Rata-rata selisih 9.9 kg

[C] PERFORMA PER TIPE GOAL
             count  Akurasi (%)
User_Goal                      
Maintain         1          0.0
Muscle Gain     12        100.0
Weight Loss      7        100.0

[D] 3 REKOMENDASI PALING MELESET (Berdasarkan Beda Berat)
  User_Goal  Teman_Goal  Selisih_Berat  Sama_Level
Muscle Gain Muscle Gain             36        True
Muscle Gain Muscle Gain             20        True
Weight Loss Weight Loss             20        True



In [45]:
data_workout = {
    'knn_model': knn,       # Model "Otak" pencari teman
    'scaler': scaler,             # [PENTING] Alat untuk mengubah input user baru jadi skala 0-1
    'profiles_db': df_profiles,   # Database daftar orang (untuk ambil ID/Nama)
    'schedule_db': df_final,      # Database jadwal (untuk ambil detail latihan)
    'encoders': {                 # Kamus penerjemah (Teks -> Angka)
        'gender': le_gender,
        'goal': le_goal,
        'level': le_level
    },
    'features': features_knn      # Urutan kolom input (supaya tidak tertukar posisi)
}

In [46]:
with open('../../models/model_workout.pickle', 'wb') as f:
        pickle.dump(data_workout, f)